# 🌍 AfriOmics | Multi-Omics Integration & Disease Modelling
## Notebook 04 — Integrating All Three Omics Layers
---
**What this notebook does:**
1. Harmonise metagenomics, metatranscriptomics, and metabolomics
2. Multi-omics factor analysis (MOFA+ equivalent in Python)
3. Build cross-omics correlation networks
4. Model cross-body-site microbiome interactions
5. Train a Random Forest disease classifier
6. Extract disease-specific multi-omics signatures
7. Visualise the complete integrated network

> 💡 **Key concept:** No single omics layer tells the full story. Integration reveals emergent patterns — microbial species × active pathways × metabolite outputs — that predict disease better than any layer alone.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'pandas', 'numpy', 'matplotlib', 'seaborn', 'plotly',
                'scipy', 'scikit-learn', 'networkx', 'pyvis'], check=True)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import networkx as nx
from pyvis.network import Network
from scipy import stats
from scipy.spatial.distance import braycurtis
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (classification_report, roc_auc_score,
                              confusion_matrix, roc_curve, auc)
from sklearn.inspection import permutation_importance
import warnings; warnings.filterwarnings('ignore')
from pathlib import Path

print('✅ AfriOmics Integration & Disease Modelling Notebook v1.0')
print('   All packages loaded.')

In [ ]:
CONFIG = {
    'results_dir'   : 'results/integration/',
    'metadata_file' : 'config/samples.tsv',
    'group_col'     : 'disease',
    'ref_level'     : 'healthy',
    'corr_threshold': 0.35,
    'pval_threshold': 0.05,
    'n_factors'     : 8,
    'seed'          : 42,
    'n_trees'       : 500,
    'n_folds'       : 5,
}

for d in ['harmonised', 'pca', 'correlations', 'network', 'disease_model', 'body_site', 'disease_signatures']:
    Path(f"{CONFIG['results_dir']}/{d}").mkdir(parents=True, exist_ok=True)

np.random.seed(CONFIG['seed'])
print('✅ Integration module configured.')

---
## STEP 1 — Data Harmonisation
**What:** Align all three omics matrices to a common sample set, normalise, and filter low-prevalence features.

**Why:** Before integration, each layer must be on a comparable scale. We use centred log-ratio (CLR) transformation for compositional microbiome data, and log-normalisation for metabolomics.

In [ ]:
def clr_transform(df, pseudocount=1e-6):
    """
    Centred log-ratio transformation for compositional data.
    Addresses compositional constraint of relative abundance data.
    """
    df = df + pseudocount
    log_df  = np.log(df)
    clr_df  = log_df.subtract(log_df.mean(axis=0), axis=1)
    return clr_df


def harmonise_omics(mg_df, mt_df, mb_df, metadata_df,
                    min_prevalence=0.2, min_abundance=0.001):
    """
    Harmonise three omics matrices:
    1. Find samples present in all three layers
    2. Filter low-prevalence features
    3. Apply appropriate normalisation per layer
    4. Return aligned matrices + QC report

    Parameters:
    -----------
    mg_df  : species × samples (metagenomics, relative abundance)
    mt_df  : pathways × samples (metatranscriptomics, CPM)
    mb_df  : metabolites × samples (metabolomics, normalised intensity)
    """
    # Find common samples
    common = list(
        set(mg_df.columns) & set(mt_df.columns) &
        set(mb_df.columns) & set(metadata_df.index)
    )
    print(f'Common samples across all 3 omics layers: {len(common)}')

    # Filter features
    def filter_features(df, samples, min_prev, min_abund):
        d = df[samples].copy()
        # Prevalence: fraction of samples where feature > min_abundance
        prevalence = (d > min_abund).sum(axis=1) / len(samples)
        return d[prevalence >= min_prev]

    mg_filt = filter_features(mg_df, common, min_prevalence, min_abundance)
    mt_filt = filter_features(mt_df, common, min_prevalence, 0.0)
    mb_filt = filter_features(mb_df, common, min_prevalence, 0.0)

    print(f'After filtering:')
    print(f'  Metagenomics  : {mg_filt.shape[0]} features')
    print(f'  Metatranscript: {mt_filt.shape[0]} features')
    print(f'  Metabolomics  : {mb_filt.shape[0]} features')

    # Normalise
    mg_norm = clr_transform(mg_filt)                     # CLR for compositional
    mt_norm = np.log1p(mt_filt)                          # log1p for count data
    mb_norm = (mb_filt - mb_filt.mean(axis=1).values.reshape(-1,1)) / \
               mb_filt.std(axis=1).values.reshape(-1,1)  # z-score for metabolomics

    # QC report
    qc = pd.DataFrame({
        'sample_id'       : common,
        'mg_n_features'   : (mg_filt[common] > 0).sum().values,
        'mt_n_features'   : (mt_filt[common] > 0).sum().values,
        'mb_n_features'   : (mb_filt[common] > 0).sum().values,
        'mg_total_abund'  : mg_filt[common].sum().values,
    })
    qc = qc.merge(metadata_df.reset_index().rename(columns={'index':'sample_id'}), on='sample_id', how='left')

    return mg_norm, mt_norm, mb_norm, qc, common

print('✅ Harmonisation functions ready.')

---
## STEP 2 — Joint Multi-Omics PCA
**What:** Concatenate all normalised omics features into one matrix, then run PCA to see how samples cluster.

**Why:** Reveals whether disease groups or African regions are the dominant sources of variation — and whether that variation is driven by one omics layer or shared across all three.

In [ ]:
def run_joint_pca(mg_norm, mt_norm, mb_norm, metadata_df, common_samples,
                  n_components=10):
    """
    Concatenate all three normalised omics matrices and run PCA.
    Returns PCA coordinates + variance explained.
    """
    # Transpose: samples × features for each layer
    X_mg = mg_norm[common_samples].T
    X_mt = mt_norm[common_samples].T
    X_mb = mb_norm[common_samples].T

    # Prefix column names to track omics layer in loadings
    X_mg.columns = [f'MG__{c}' for c in X_mg.columns]
    X_mt.columns = [f'MT__{c}' for c in X_mt.columns]
    X_mb.columns = [f'MB__{c}' for c in X_mb.columns]

    X_joint = pd.concat([X_mg, X_mt, X_mb], axis=1)
    X_joint = X_joint.fillna(0)

    # Scale
    scaler    = StandardScaler()
    X_scaled  = scaler.fit_transform(X_joint)

    # PCA
    pca = PCA(n_components=min(n_components, X_scaled.shape[1]), random_state=42)
    coords = pca.fit_transform(X_scaled)
    var_exp = np.round(pca.explained_variance_ratio_ * 100, 2)

    pca_df = pd.DataFrame(
        coords,
        index   = common_samples,
        columns = [f'PC{i+1}' for i in range(coords.shape[1])]
    )
    pca_df = pca_df.join(metadata_df.loc[common_samples])

    return pca_df, var_exp, pca


def plot_joint_pca(pca_df, var_exp, colour_col='disease', symbol_col='body_site'):
    """Interactive joint multi-omics PCA scatter plot."""
    symbol_arg = symbol_col if symbol_col in pca_df.columns else None

    fig = px.scatter(
        pca_df.reset_index(), x='PC1', y='PC2',
        color  = colour_col if colour_col in pca_df.columns else None,
        symbol = symbol_arg,
        hover_name = 'index',
        hover_data = [c for c in ['disease', 'body_site', 'region'] if c in pca_df.columns],
        title  = f'Joint Multi-Omics PCA<br><sup>PC1: {var_exp[0]}% | PC2: {var_exp[1]}% variance explained</sup>',
        labels = {'PC1': f'PC1 ({var_exp[0]}%)', 'PC2': f'PC2 ({var_exp[1]}%)'},
        template = 'plotly_dark',
        color_discrete_sequence = px.colors.qualitative.Set2
    )
    fig.update_traces(marker=dict(size=12, opacity=0.85))
    fig.update_layout(
        paper_bgcolor='#111710', plot_bgcolor='#111710', font_color='#e8f0e9'
    )
    return fig

print('✅ Joint PCA functions ready.')

---
## STEP 3 — Random Forest Disease Classifier
**What:** Train a machine learning model using integrated multi-omics features to predict disease state.

**Why:** A multi-omics model outperforms any single layer. The features selected by the model constitute the disease's multi-omics signature.

**African diseases modelled:** malaria · HIV · TB · healthy

In [ ]:
def train_multiomics_rf(mg_norm, mt_norm, mb_norm, metadata_df, common_samples,
                         group_col='disease', n_estimators=500, n_folds=5, seed=42):
    """
    Random Forest classifier from integrated multi-omics features.
    Uses stratified k-fold cross-validation.

    Returns:
    --------
    model        : trained RandomForestClassifier
    cv_scores    : cross-validated accuracy scores
    importance_df: feature importance DataFrame
    predictions  : out-of-fold predictions (for ROC curves)
    """
    # Build feature matrix
    X_mg = mg_norm[common_samples].T
    X_mt = mt_norm[common_samples].T
    X_mb = mb_norm[common_samples].T
    X_mg.columns = [f'MG__{c}' for c in X_mg.columns]
    X_mt.columns = [f'MT__{c}' for c in X_mt.columns]
    X_mb.columns = [f'MB__{c}' for c in X_mb.columns]

    X = pd.concat([X_mg, X_mt, X_mb], axis=1).fillna(0)

    # Labels
    y = metadata_df.loc[common_samples, group_col]
    le = LabelEncoder()
    y_enc = le.fit_transform(y)
    classes = le.classes_

    print(f'Training Random Forest:')
    print(f'  Samples   : {len(common_samples)}')
    print(f'  Features  : {X.shape[1]}')
    print(f'  Classes   : {list(classes)}')
    print(f'  CV folds  : {n_folds}')
    print(f'  N trees   : {n_estimators}')

    # Cross-validation
    skf    = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    rf     = RandomForestClassifier(n_estimators=n_estimators, random_state=seed,
                                     n_jobs=-1, class_weight='balanced')

    cv_scores  = cross_val_score(rf, X, y_enc, cv=skf, scoring='balanced_accuracy')
    cv_preds   = cross_val_predict(rf, X, y_enc, cv=skf, method='predict_proba')

    print(f'\nCV Balanced Accuracy: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}')

    # Train final model on all data
    rf.fit(X, y_enc)

    # Feature importance
    imp_df = pd.DataFrame({
        'feature'     : X.columns,
        'importance'  : rf.feature_importances_,
        'omics_layer' : [c.split('__')[0] for c in X.columns]
    }).sort_values('importance', ascending=False)

    return rf, cv_scores, imp_df, cv_preds, le, y_enc, classes


def plot_roc_curves(cv_preds, y_enc, classes):
    """Plot one-vs-rest ROC curves for each disease class."""
    disease_colours = {
        'healthy'  : '#4aff91', 'malaria': '#ff6b6b',
        'HIV'      : '#f5c842', 'TB': '#5bc4ff',
        'cholera'  : '#fb923c', 'schistosomiasis': '#c084fc'
    }

    fig = go.Figure()
    for i, cls in enumerate(classes):
        y_bin   = (y_enc == i).astype(int)
        fpr, tpr, _ = roc_curve(y_bin, cv_preds[:, i])
        auroc   = auc(fpr, tpr)
        colour  = disease_colours.get(cls, '#7a9b7e')

        fig.add_trace(go.Scatter(
            x=fpr, y=tpr,
            name  = f'{cls} (AUROC={auroc:.3f})',
            mode  = 'lines',
            line  = dict(color=colour, width=2.5)
        ))

    fig.add_trace(go.Scatter(
        x=[0, 1], y=[0, 1], mode='lines',
        line=dict(color='#3a4a3a', dash='dash'),
        name='Random (AUROC=0.500)', showlegend=True
    ))

    fig.update_layout(
        title = 'ROC Curves — Multi-Omics Disease Classifier<br><sup>AfriOmics | Cross-validated predictions</sup>',
        xaxis = dict(title='False Positive Rate', gridcolor='#2a3828', range=[0, 1]),
        yaxis = dict(title='True Positive Rate', gridcolor='#2a3828', range=[0, 1]),
        paper_bgcolor='#111710', plot_bgcolor='#111710', font_color='#e8f0e9',
        width=700, height=550
    )
    return fig


def plot_feature_importance(imp_df, top_n=40):
    """Plot top features coloured by omics layer."""
    layer_colours = {'MG': '#4aff91', 'MT': '#5bc4ff', 'MB': '#f5c842'}
    top = imp_df.head(top_n).copy()
    top['feature_clean'] = top['feature'].str.replace(r'^(MG|MT|MB)__', '', regex=True)
    top['colour']        = top['omics_layer'].map(layer_colours)

    fig = px.bar(
        top.sort_values('importance'),
        x       = 'importance',
        y       = 'feature_clean',
        color   = 'omics_layer',
        color_discrete_map = layer_colours,
        orientation = 'h',
        title   = f'Top {top_n} Multi-Omics Disease Features (Random Forest)',
        labels  = {'importance': 'Mean Decrease Impurity', 'feature_clean': ''},
        template = 'plotly_dark'
    )
    fig.update_layout(
        paper_bgcolor='#111710', plot_bgcolor='#111710', font_color='#e8f0e9',
        height=max(400, top_n * 18)
    )
    return fig

print('✅ Random Forest functions ready.')

---
## STEP 4 — Multi-Omics Interaction Network
**What:** Build a network where nodes are taxa, active pathways, and metabolites; edges are significant correlations.

**Why:** Networks reveal hub nodes (key taxa that connect many metabolites), bridges between omics layers, and disease-specific sub-networks.

**Visualised as an interactive HTML graph you can explore in a browser.**

In [ ]:
def compute_cross_omics_correlations(mg_norm, mt_norm, mb_norm, common_samples,
                                      corr_threshold=0.35, pval_threshold=0.05,
                                      top_n_features=30):
    """
    Compute pairwise Spearman correlations between:
    - Top metagenomics features vs metabolomics features
    - Top metatranscriptomics features vs metabolomics features

    Returns a DataFrame of significant edges.
    """
    from scipy.stats import spearmanr

    # Select top variable features per layer
    def top_var(df, samples, n):
        sub = df[samples]
        return sub.loc[sub.var(axis=1).nlargest(n).index]

    mg_top = top_var(mg_norm, common_samples, top_n_features)
    mt_top = top_var(mt_norm, common_samples, top_n_features)
    mb_top = top_var(mb_norm, common_samples, top_n_features)

    edges = []

    # MG ↔ MB
    print('Computing taxa ↔ metabolite correlations...')
    for taxon in mg_top.index:
        for metabolite in mb_top.index:
            x = mg_top.loc[taxon, common_samples].values
            y = mb_top.loc[metabolite, common_samples].values
            r, p = spearmanr(x, y)
            if abs(r) >= corr_threshold and p < pval_threshold:
                edges.append({'from': taxon, 'to': metabolite, 'correlation': r,
                              'pvalue': p, 'edge_type': 'taxa_metabolite',
                              'from_layer': 'MG', 'to_layer': 'MB'})

    # MT ↔ MB
    print('Computing active pathway ↔ metabolite correlations...')
    for pathway in mt_top.index:
        for metabolite in mb_top.index:
            x = mt_top.loc[pathway, common_samples].values
            y = mb_top.loc[metabolite, common_samples].values
            r, p = spearmanr(x, y)
            if abs(r) >= corr_threshold and p < pval_threshold:
                edges.append({'from': pathway, 'to': metabolite, 'correlation': r,
                              'pvalue': p, 'edge_type': 'pathway_metabolite',
                              'from_layer': 'MT', 'to_layer': 'MB'})

    # MG ↔ MT
    print('Computing taxa ↔ active pathway correlations...')
    for taxon in mg_top.index:
        for pathway in mt_top.index:
            x = mg_top.loc[taxon, common_samples].values
            y = mt_top.loc[pathway, common_samples].values
            r, p = spearmanr(x, y)
            if abs(r) >= corr_threshold and p < pval_threshold:
                edges.append({'from': taxon, 'to': pathway, 'correlation': r,
                              'pvalue': p, 'edge_type': 'taxa_pathway',
                              'from_layer': 'MG', 'to_layer': 'MT'})

    edge_df = pd.DataFrame(edges)
    print(f'\nTotal significant edges: {len(edge_df)}')
    if len(edge_df) > 0:
        print(edge_df['edge_type'].value_counts().to_string())
    return edge_df


def build_and_visualise_network(edge_df, output_html='network.html'):
    """
    Build an interactive network from cross-omics edges.
    Saves an interactive HTML file.
    """
    if edge_df.empty:
        print('No edges to visualise.')
        return None

    node_colours = {'MG': '#4aff91', 'MT': '#5bc4ff', 'MB': '#f5c842'}
    edge_colours = {'positive': '#4aff91', 'negative': '#ff6b6b'}

    G = nx.Graph()

    # Add nodes with metadata
    all_nodes = set(edge_df['from']) | set(edge_df['to'])
    node_layer = {}
    for _, row in edge_df.iterrows():
        node_layer[row['from']] = row['from_layer']
        node_layer[row['to']]   = row['to_layer']

    for node in all_nodes:
        G.add_node(node, layer=node_layer.get(node, 'Unknown'))

    # Add edges
    for _, row in edge_df.iterrows():
        G.add_edge(row['from'], row['to'],
                   weight=abs(row['correlation']),
                   direction='positive' if row['correlation'] > 0 else 'negative',
                   edge_type=row['edge_type'])

    # pyvis interactive visualisation
    net = Network(height='700px', width='100%',
                  bgcolor='#0a0e0c', font_color='#e8f0e9')
    net.from_nx(G)

    for node in net.nodes:
        layer   = node_layer.get(node['id'], 'Unknown')
        degree  = G.degree(node['id'])
        node['color'] = node_colours.get(layer, '#7a9b7e')
        node['size']  = max(10, min(40, degree * 4))
        node['title'] = f"<b>{node['id']}</b><br>Layer: {layer}<br>Degree: {degree}"
        node['label'] = node['id'] if degree >= 3 else ''
        node['font']  = {'color': '#e8f0e9', 'size': 12}

    for edge in net.edges:
        direction = G.edges[edge['from'], edge['to']].get('direction', 'positive')
        edge['color']  = edge_colours[direction]
        edge['width']  = max(1, G.edges[edge['from'], edge['to']].get('weight', 0.5) * 3)
        edge['smooth'] = True

    net.set_options("""
    var options = {
      physics: {
        forceAtlas2Based: {gravitationalConstant: -50, springLength: 120},
        solver: 'forceAtlas2Based',
        stabilization: {iterations: 150}
      },
      interaction: {hover: true, tooltipDelay: 100}
    }""")

    net.save_graph(output_html)
    print(f'✅ Interactive network saved to {output_html}')
    print(f'   Nodes: {G.number_of_nodes()} | Edges: {G.number_of_edges()}')

    return G

print('✅ Network functions ready.')

---
## ▶ RUN ALL — Full Integration Pipeline (Demo)

In [ ]:
print('=' * 65)
print('AfriOmics | Multi-Omics Integration — Demo Mode')
print('=' * 65)

np.random.seed(42)
N_SAMPLES = 30

sample_names = [f'AFRI_{str(i).zfill(3)}' for i in range(1, N_SAMPLES + 1)]

disease_labels = (['malaria'] * 8 + ['HIV'] * 8 + ['TB'] * 7 + ['healthy'] * 7)
region_labels  = (['west_africa'] * 8 + ['east_africa'] * 8 +
                   ['southern_africa'] * 7 + ['central_africa'] * 7)

metadata_demo = pd.DataFrame({
    'disease'      : disease_labels,
    'body_site'    : ['gut'] * N_SAMPLES,
    'region'       : region_labels,
    'diet_type'    : np.random.choice(['traditional', 'mixed', 'western'], N_SAMPLES),
    'urbanisation' : np.random.choice(['rural', 'urban'], N_SAMPLES)
}, index=sample_names)

# Synthetic omics data
species = [f'Species_{i}' for i in range(60)]
pathways = [f'Pathway_{i}' for i in range(50)]
metabolites = [f'Metabolite_{i}' for i in range(55)]

mg_demo = pd.DataFrame(np.random.dirichlet(np.ones(60)*0.5, N_SAMPLES).T * 100,
                        index=species, columns=sample_names)
mt_demo = pd.DataFrame(np.random.lognormal(2, 1, (50, N_SAMPLES)),
                        index=pathways, columns=sample_names)
mb_demo = pd.DataFrame(np.random.lognormal(5, 1.5, (55, N_SAMPLES)),
                        index=metabolites, columns=sample_names)

# Inject disease signal
malaria_idx = [s for s in sample_names if disease_labels[sample_names.index(s)] == 'malaria']
mg_demo.loc['Species_0', malaria_idx] *= 8
mt_demo.loc['Pathway_0', malaria_idx] *= 6
mb_demo.loc['Metabolite_0', malaria_idx] *= 5
mb_demo.loc['Metabolite_1', malaria_idx] *= 0.1

# ---- HARMONISE ----
print('\n[1/4] Harmonising omics layers...')
mg_norm, mt_norm, mb_norm, qc_df, common = harmonise_omics(
    mg_demo, mt_demo, mb_demo, metadata_demo)

# ---- JOINT PCA ----
print('\n[2/4] Running joint multi-omics PCA...')
pca_df, var_exp, pca_model = run_joint_pca(mg_norm, mt_norm, mb_norm, metadata_demo, common)
print(f'   Variance: PC1={var_exp[0]}% | PC2={var_exp[1]}% | PC3={var_exp[2]}%')

fig_pca = plot_joint_pca(pca_df, var_exp, colour_col='disease')
fig_pca.show()

pca_df.to_csv(f"{CONFIG['results_dir']}/pca/joint_pca_coordinates.tsv", sep='\t')

# ---- RANDOM FOREST ----
print('\n[3/4] Training Random Forest disease classifier...')
rf_model, cv_scores, imp_df, cv_preds, le, y_enc, classes = train_multiomics_rf(
    mg_norm, mt_norm, mb_norm, metadata_demo, common,
    group_col='disease', n_estimators=CONFIG['n_trees'],
    n_folds=CONFIG['n_folds'], seed=CONFIG['seed']
)

fig_roc = plot_roc_curves(cv_preds, y_enc, classes)
fig_roc.show()

fig_imp = plot_feature_importance(imp_df, top_n=30)
fig_imp.show()

imp_df.to_csv(f"{CONFIG['results_dir']}/disease_model/feature_importance.tsv", sep='\t', index=False)

# ---- NETWORK ----
print('\n[4/4] Building multi-omics interaction network...')
edge_df = compute_cross_omics_correlations(
    mg_norm, mt_norm, mb_norm, common,
    corr_threshold=CONFIG['corr_threshold'],
    pval_threshold=CONFIG['pval_threshold'],
    top_n_features=20
)

net_html = f"{CONFIG['results_dir']}/network/multiomics_network.html"
G = build_and_visualise_network(edge_df, output_html=net_html)

edge_df.to_csv(f"{CONFIG['results_dir']}/network/network_edges.tsv", sep='\t', index=False)

print('\n' + '=' * 65)
print('✅ AfriOmics Integration Complete!')
print('Results saved to results/integration/')
print('\nNext step: Generate the unified report')
print('  snakemake --snakefile workflow/Snakefile_integration results/integration/report/afriomics_integrated_report.html')

---
## STEP 5 — Disease-Specific Multi-Omics Signatures
**What:** Extract the characteristic multi-omics fingerprint for each disease. Which taxa, pathways, and metabolites together define malaria vs HIV vs TB vs healthy?

**Output:** One signature panel per disease — exportable as a publication figure.

In [ ]:
def extract_disease_signature(mg_norm, mt_norm, mb_norm, metadata_demo,
                               imp_df, common, disease, top_n=10):
    """
    Extract the top multi-omics features defining a specific disease.
    Shows mean feature values in disease vs healthy for each omics layer.
    """
    dis_samples     = [s for s in common if metadata_demo.loc[s, 'disease'] == disease]
    healthy_samples = [s for s in common if metadata_demo.loc[s, 'disease'] == 'healthy']

    if not dis_samples or not healthy_samples:
        print(f'Not enough samples for {disease}')
        return None

    # Top features from RF importance for this disease
    top_mg_feats = imp_df[imp_df['omics_layer'] == 'MG'].head(top_n)['feature'].str.replace('MG__','').tolist()
    top_mt_feats = imp_df[imp_df['omics_layer'] == 'MT'].head(top_n)['feature'].str.replace('MT__','').tolist()
    top_mb_feats = imp_df[imp_df['omics_layer'] == 'MB'].head(top_n)['feature'].str.replace('MB__','').tolist()

    def sig_compare(df, feats, dis_s, hlt_s):
        feats = [f for f in feats if f in df.index]
        if not feats: return pd.DataFrame()
        rows = []
        for f in feats:
            d_val = df.loc[f, dis_s].mean()
            h_val = df.loc[f, hlt_s].mean()
            fc    = d_val - h_val   # CLR-normalised, so difference = log ratio
            _, p  = stats.mannwhitneyu(
                df.loc[f, dis_s].values,
                df.loc[f, hlt_s].values, alternative='two-sided'
            )
            rows.append({'feature': f, 'disease_mean': d_val, 'healthy_mean': h_val,
                         'log_fold_change': fc, 'pvalue': p})
        return pd.DataFrame(rows).sort_values('log_fold_change', ascending=False)

    mg_sig = sig_compare(mg_norm, top_mg_feats, dis_samples, healthy_samples)
    mt_sig = sig_compare(mt_norm, top_mt_feats, dis_samples, healthy_samples)
    mb_sig = sig_compare(mb_norm, top_mb_feats, dis_samples, healthy_samples)

    # Combined signature panel
    fig = make_subplots(rows=1, cols=3,
        subplot_titles=['Microbial Taxa (MG)', 'Active Pathways (MT)', 'Metabolites (MB)'],
        horizontal_spacing=0.08)

    colour = {'up': '#ff6b6b', 'down': '#5bc4ff'}

    for col_i, (sig_df, label) in enumerate([
        (mg_sig, 'MG'), (mt_sig, 'MT'), (mb_sig, 'MB')]):
        if sig_df.empty: continue
        bar_colours = ['#ff6b6b' if v > 0 else '#5bc4ff' for v in sig_df['log_fold_change']]
        fig.add_trace(go.Bar(
            x=sig_df['log_fold_change'], y=sig_df['feature'],
            orientation='h', marker_color=bar_colours,
            name=label, showlegend=False
        ), row=1, col=col_i+1)

    fig.update_layout(
        title=f'Multi-Omics Signature: {disease.upper()} vs Healthy<br><sup>AfriOmics | Red = enriched in disease; Blue = depleted</sup>',
        paper_bgcolor='#111710', plot_bgcolor='#111710', font_color='#e8f0e9',
        height=450, width=1200
    )
    fig.update_xaxes(gridcolor='#2a3828', zerolinecolor='#f5c842')
    fig.update_yaxes(gridcolor='#2a3828')

    return fig, mg_sig, mt_sig, mb_sig

# Run for each disease
print('Generating disease-specific multi-omics signatures...\n')
for disease in ['malaria', 'HIV', 'TB']:
    result = extract_disease_signature(
        mg_norm, mt_norm, mb_norm, metadata_demo, imp_df, common, disease, top_n=8
    )
    if result:
        fig_sig, mg_s, mt_s, mb_s = result
        fig_sig.show()
        mg_s.to_csv(f"{CONFIG['results_dir']}/disease_signatures/{disease}_metagenomics.tsv", sep='\t', index=False)
        mt_s.to_csv(f"{CONFIG['results_dir']}/disease_signatures/{disease}_metatranscriptomics.tsv", sep='\t', index=False)
        mb_s.to_csv(f"{CONFIG['results_dir']}/disease_signatures/{disease}_metabolomics.tsv", sep='\t', index=False)
        print(f'✅ {disease} signature saved.')

print('\n✅ All disease signatures complete.')
print('\nOpen results/integration/network/multiomics_network.html for the interactive network.')